# Vision Transformer (ViT) — Crop Leaf Disease Detection v2
**Team 179 | B.Tech Research Project**

This notebook fine-tunes **ViT-Base** on a **combined PlantDoc + PlantVillage dataset** to improve accuracy on hard classes.

### Why combine datasets?
- PlantDoc v1 had low accuracy on blight/bacterial/spot classes due to too few training samples
- PlantVillage adds 54,000+ images across 38 classes — massively boosting the hard classes
- Combined dataset: ~56,000+ images, covering all 27 PlantDoc classes with far more examples

### Datasets
**PlantDoc** — 2,569 images, 27 classes, real field conditions (IIT, CoDS-COMAD 2020)
- Add: search **'plant-doc-dataset'** (abdulhasibuddin) in + Add Data

**PlantVillage** — 54,303 images, 38 classes, controlled lab conditions
- Add: search **'plantvillage-dataset'** (abdallahalidev) in + Add Data

### Hardware
- Kaggle T4 GPU (16GB VRAM)
- Estimated training time: **2–3 hours** (larger dataset)


## Step 1: Install Dependencies

In [ ]:
# Install HuggingFace Transformers + supporting libraries
!pip install transformers==4.40.0 --quiet
!pip install datasets accelerate timm --quiet
!pip install scikit-learn matplotlib seaborn --quiet

import torch
import transformers
print(f'PyTorch     : {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## Step 2: Load PlantDoc + PlantVillage Datasets

> **Before running, add BOTH datasets via + Add Data:**
> 1. Search **`plant-doc-dataset`** by abdulhasibuddin → Add
> 2. Search **`plantvillage-dataset`** by abdallahalidev → Add

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from collections import Counter

# ─── Locate both datasets ────────────────────────────────────────────────────
PLANTDOC_BASE    = None
PLANTVILLAGE_BASE = None

for p in Path('/kaggle/input').iterdir():
    name = p.name.lower()
    if 'plantvillage' in name or 'plant-village' in name or 'plant_village' in name:
        PLANTVILLAGE_BASE = p
    elif 'plantdoc' in name or 'plant-doc' in name or 'plant_doc' in name:
        PLANTDOC_BASE = p

print('Available datasets in /kaggle/input:')
for p in Path('/kaggle/input').iterdir():
    print(f'  {p.name}')

if PLANTDOC_BASE is None:
    raise FileNotFoundError('PlantDoc not found. Add abdulhasibuddin/plant-doc-dataset via + Add Data.')
if PLANTVILLAGE_BASE is None:
    raise FileNotFoundError('PlantVillage not found. Add abdallahalidev/plantvillage-dataset via + Add Data.')

print(f'\nPlantDoc path    : {PLANTDOC_BASE}')
print(f'PlantVillage path: {PLANTVILLAGE_BASE}')

In [ ]:
# ─── PlantVillage → PlantDoc class name mapping ──────────────────────────────
# PlantVillage uses names like 'Tomato___Early_blight'
# PlantDoc uses names like 'Tomato Early blight leaf'
# This map normalises PlantVillage class names to match PlantDoc exactly

PV_TO_PLANTDOC = {
    # Tomato
    'Tomato___Early_blight':              'Tomato Early blight leaf',
    'Tomato___Late_blight':               'Tomato leaf late blight',
    'Tomato___Bacterial_spot':            'Tomato leaf bacterial spot',
    'Tomato___Septoria_leaf_spot':        'Tomato Septoria leaf spot',
    'Tomato___Leaf_Mold':                 'Tomato mold leaf',
    'Tomato___Tomato_mosaic_virus':       'Tomato leaf mosaic virus',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'Tomato leaf yellow virus',
    'Tomato___healthy':                   'Tomato leaf',
    'Tomato___Spider_mites Two-spotted_spider_mite': 'Tomato leaf',
    'Tomato___Target_Spot':               'Tomato leaf',

    # Potato
    'Potato___Early_Blight':              'Potato leaf early blight',
    'Potato___Early_blight':              'Potato leaf early blight', # Added lowercase
    'Potato___Late_Blight':               'Potato leaf late blight',
    'Potato___Late_blight':               'Potato leaf late blight', # Added lowercase
    'Potato___healthy':                   'Potato leaf',

    # Apple
    'Apple___Apple_scab':                 'Apple Scab Leaf',
    'Apple___healthy':                    'Apple leaf',
    'Apple___Cedar_apple_rust':           'Apple rust leaf',
    'Apple___Black_rot':                  'Apple leaf',

    # Grape
    'Grape___healthy':                    'grape leaf',
    'Grape___Black_rot':                  'grape leaf black rot',
    'Grape___Esca_(Black_Measles)':       'grape leaf', 
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 'grape leaf',

    # Corn / Maize
    'Corn_(maize)___Gray_leaf_spot':      'Corn Gray leaf spot',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Corn Gray leaf spot', # Added
    'Corn_(maize)___Northern_Leaf_Blight':'Corn leaf blight',
    'Corn_(maize)___Common_rust_':         'Corn rust leaf',
    'Corn_(maize)___healthy':             'Corn leaf',

    # Bell Pepper
    'Pepper,_bell___healthy':             'Bell_pepper leaf',
    'Pepper,_bell___Bacterial_spot':      'Bell_pepper leaf spot',

    # Others
    'Strawberry___healthy':               'Strawberry leaf',
    'Strawberry___Leaf_scorch':           'Strawberry leaf',
    'Raspberry___healthy':                'Raspberry leaf',
    'Blueberry___healthy':                'Blueberry leaf',
    'Soybean___healthy':                  'Soyabean leaf',
    'Squash___Powdery_mildew':            'Squash Powdery mildew leaf',
    'Peach___healthy':                    'Peach leaf',
    'Peach___Bacterial_spot':             'Peach leaf',
    'Cherry_(including_sour)___healthy':  'Cherry leaf',
    'Cherry_(including_sour)___Powdery_mildew': 'Cherry leaf',
    'Orange___Haunglongbing_(Citrus_greening)': 'Orange leaf' # New
}

print(f'Mapping covers {len(PV_TO_PLANTDOC)} PlantVillage classes → PlantDoc names')

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import json
from pathlib import Path
from collections import Counter

# ─── THE SPEED UP: Fast, case-insensitive image finder ────────────────────────
def get_images_from_dir(directory):
    """Blazing fast image retrieval that catches .jpg, .JPG, .jpeg, and .png"""
    valid_extensions = {'.jpg', '.jpeg', '.png'}
    # iterdir() is 10x faster than glob()
    return [
        str(p) for p in directory.iterdir() 
        if p.is_file() and p.suffix.lower() in valid_extensions
    ]
# ──────────────────────────────────────────────────────────────────────────────

def load_plantdoc(base):
    samples = []
    base = Path(base)
    
    if (base / 'PlantDoc-Dataset').exists():
        base = base / 'PlantDoc-Dataset'
        
    split_found = False
    for split in ['train', 'test', 'val', 'valid']:
        split_dir = base / split
        if split_dir.exists():
            split_found = True
            for class_dir in sorted(split_dir.iterdir()):
                if class_dir.is_dir():
                    for img_path in get_images_from_dir(class_dir):
                        samples.append((img_path, class_dir.name, 'plantdoc'))
                        
    if not split_found:
        for class_dir in sorted(base.iterdir()):
            if class_dir.is_dir() and class_dir.name not in ['train', 'test', 'val']:
                for img_path in get_images_from_dir(class_dir):
                    samples.append((img_path, class_dir.name, 'plantdoc'))
    return samples

def load_plantvillage(base, name_map):
    samples = []
    base = Path(base)
    unmapped_classes = []
    
    # Kaggle's PlantVillage often hides inside 'plantvillage dataset/color'
    search_roots = [base]
    for sub in ['color', 'train', 'plantvillage', 'PlantVillage', 'plantvillage dataset/color']:
        if (base / sub).exists():
            search_roots = [base / sub]
            break
            
    for root in search_roots:
        for class_dir in sorted(root.iterdir()):
            if not class_dir.is_dir():
                continue
                
            pv_name = class_dir.name
            
            if pv_name in name_map:
                final_class_name = name_map[pv_name]
            else:
                final_class_name = pv_name
                if pv_name not in unmapped_classes:
                    unmapped_classes.append(pv_name)
                    
            for img_path in get_images_from_dir(class_dir):
                samples.append((img_path, final_class_name, 'plantvillage'))
                
    if unmapped_classes:
        print(f"  Note: {len(unmapped_classes)} PlantVillage classes were not in your map.")
        
    return samples

# Load both datasets
print('Loading PlantDoc...')
plantdoc_samples = load_plantdoc(PLANTDOC_BASE)
print(f'  PlantDoc: {len(plantdoc_samples)} images')

print('Loading PlantVillage...')
plantvillage_samples = load_plantvillage(PLANTVILLAGE_BASE, PV_TO_PLANTDOC)
print(f'  PlantVillage (mapped): {len(plantvillage_samples)} images')

# Combine
all_samples = plantdoc_samples + plantvillage_samples
print(f'\nCombined total: {len(all_samples)} images')

# Build class index from PlantDoc class names (ground truth)
class_names   = sorted(list(set([s[1] for s in all_samples])))
class_to_idx  = {c: i for i, c in enumerate(class_names)}

print(f'Number of classes: {len(class_names)}')
print(f'\nPer-class image counts (PlantDoc + PlantVillage):')
plantdoc_counts   = Counter([s[1] for s in plantdoc_samples])
plantvillage_counts = Counter([s[1] for s in plantvillage_samples])
for c in class_names:
    pd_count  = plantdoc_counts.get(c, 0)
    pv_count  = plantvillage_counts.get(c, 0)
    print(f'  {c[:35]:<35} PlantDoc: {pd_count:>4}  |  PlantVillage: {pv_count:>5}  |  Total: {pd_count+pv_count}')

with open('/kaggle/working/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
print('\nClass names saved!')

In [ ]:
# Visualize sample images per class
from PIL import Image
import matplotlib.pyplot as plt
import random

# Show 12 random samples with labels
random.seed(42)
samples_to_show = random.sample(all_samples, min(12, len(all_samples)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('PlantDoc + PlantVillage — Sample Images', fontsize=16, fontweight='bold')
axes = axes.flatten()

for ax, (img_path, label, split) in zip(axes, samples_to_show):
    try:
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        # Shorten long class names for display
        short_label = label[:25] + '...' if len(label) > 25 else label
        ax.set_title(short_label, fontsize=8, fontweight='bold')
        ax.axis('off')
    except Exception as e:
        ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample images saved!')

## Step 3: Prepare Dataset for ViT

ViT-Base expects:
- Images resized to **224×224**
- Normalized with ImageNet mean/std: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`
- This is because ViT was pretrained on ImageNet — matching its preprocessing is critical

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch
import random
from sklearn.model_selection import train_test_split

# ViT image preprocessing
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PlantDiseaseDataset(Dataset):
    def __init__(self, samples, class_to_idx, transform=None):
        self.samples = samples
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, _ = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.class_to_idx[label]

# --- THE FIX: STRICT REAL-WORLD EVALUATION SPLIT ---
print("Applying strict domain split: Validating ONLY on real-world PlantDoc images.")

# 1. Split ONLY PlantDoc into 80/10/10 
pd_labels = [s[1] for s in plantdoc_samples]
pd_train, pd_temp, _, pd_temp_labels = train_test_split(
    plantdoc_samples, pd_labels, test_size=0.2, stratify=pd_labels, random_state=42
)
pd_val, pd_test = train_test_split(
    pd_temp, test_size=0.5, stratify=pd_temp_labels, random_state=42
)

# 2. Force ALL PlantVillage images strictly into the training set
train_samples = pd_train + plantvillage_samples
val_samples   = pd_val
test_samples  = pd_test

print(f'Train: {len(train_samples)} images (PlantDoc + PlantVillage)')
print(f'Val  : {len(val_samples)} images (PlantDoc ONLY)')
print(f'Test : {len(test_samples)} images (PlantDoc ONLY)')

train_dataset = PlantDiseaseDataset(train_samples, class_to_idx, train_transform)
val_dataset   = PlantDiseaseDataset(val_samples,   class_to_idx, val_transform)
test_dataset  = PlantDiseaseDataset(test_samples,  class_to_idx, val_transform)

# Increased batch_size to 64 for Dual T4 GPUs
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nDataLoaders ready!')
print(f'Batch size: 64 | Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Step 4: Load Pretrained ViT-Base Model

We use **google/vit-base-patch16-224** — pretrained on ImageNet-21k (14 million images).

**What this means:**
- The model already knows about edges, textures, shapes, colors from ImageNet
- We just replace the final classification head with our disease classes
- Fine-tuning takes only 5–10 epochs instead of 100+ from scratch

**ViT-Base architecture:**
- Patch size: 16×16 pixels → 196 patches from a 224×224 image
- Hidden dimension: 768
- 12 transformer layers
- 12 attention heads
- 86M parameters total

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor # Updated name
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

NUM_CLASSES = len(class_names)
print(f'Number of classes: {NUM_CLASSES}')

# Load model with correct labels and ID mapping
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=NUM_CLASSES,
    id2label={i: c for i, c in enumerate(class_names)},
    label2id={c: i for i, c in enumerate(class_names)},
    ignore_mismatched_sizes=True 
)

# Explicitly set the task type for the config
model.config.problem_type = "single_label_classification"

# Move to GPU
model = model.to(device)

# Enable Multi-GPU training
if torch.cuda.device_count() > 1:
    print(f"🔥 Unleashing {torch.cuda.device_count()} GPUs!")
    # NOTE: We use DataParallel for simplicity on Kaggle
    model = nn.DataParallel(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

print(f'\nModel architecture summary:')
print(f'  Patch size    : 16×16')
print(f'  Hidden dim    : 768')
print(f'  Layers        : 12')
print(f'  Heads         : 12')
print(f'  Output classes: {NUM_CLASSES}')
print('\nModel loaded successfully!')

## Step 5: Train the ViT Model

**Training strategy:**
- Optimizer: AdamW (best for transformers)
- Learning rate: 2e-5 (low LR — transformer fine-tuning needs gentle updates)
- LR scheduler: cosine warmup (ramps up LR slowly then decays)
- Mixed precision (FP16) — halves memory usage, doubles speed on T4
- Early stopping: stops if val accuracy doesn't improve for 5 epochs

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.amp import GradScaler, autocast  # Updated to remove FutureWarnings
from transformers import get_cosine_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import time
import json

# ─── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS        = 20
LR            = 2e-5      # Low LR for transformer fine-tuning
WEIGHT_DECAY  = 0.01
WARMUP_EPOCHS = 2         # Now actually used!
PATIENCE      = 7         # Early stopping patience

# ─── Proper Warmup Scheduler ───────────────────────────────────────────
total_steps  = len(train_loader) * EPOCHS
warmup_steps = len(train_loader) * WARMUP_EPOCHS
print(f'Total training steps : {total_steps}')
print(f'Warmup steps         : {warmup_steps} ({WARMUP_EPOCHS} epochs)')

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# ─── Fix 1: Clipped Class-Weighted Loss ───────────────────────────────────────
train_labels = [class_to_idx[s[1]] for s in train_samples]
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# CLIP THE WEIGHTS to prevent exploding gradients on rare classes
weights = np.clip(weights, a_min=None, a_max=10.0)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f'Class weights computed for {len(weights)} classes')
print(f'Min weight: {weights.min():.3f} | Max weight: {weights.max():.3f}')

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# Updated to new PyTorch AMP API
scaler = GradScaler('cuda')  
print('Optimizer, scheduler, and weighted loss ready!')

# ─── Training Loop ────────────────────────────────────────────────────────────
best_val_acc  = 0.0
patience_ctr  = 0
history       = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('Starting ViT fine-tuning...')
print(f'Epochs: {EPOCHS} | LR: {LR} | Batch: 64 | Device: {device}')
print('='*65)

for epoch in range(EPOCHS):
    t0 = time.time()

    # ── TRAIN ──
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        with autocast('cuda'):  # Updated to remove FutureWarnings
            outputs = model(images).logits
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        
        # ─── Fix 2: Safe Scheduler Stepping ───────────────────────────────────
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        scale_after = scaler.get_scale()
        
        # Only step the scheduler if the scaler didn't shrink the scale (which implies a skipped step)
        if scale_before <= scale_after:
            scheduler.step()

        train_loss    += loss.item()
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

        if (batch_idx + 1) % 20 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {loss.item():.4f} | LR: {current_lr:.2e}', end='\r')

    # ── VALIDATE ──
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with autocast('cuda'): # Updated to remove FutureWarnings
                outputs = model(images).logits
                loss    = criterion(outputs, labels)
            val_loss    += loss.item()
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    # ── Metrics ──
    t_loss = train_loss / len(train_loader)
    t_acc  = train_correct / train_total * 100
    v_loss = val_loss / len(val_loader)
    v_acc  = val_correct / val_total * 100
    elapsed = time.time() - t0

    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)

    print(f'Epoch [{epoch+1:02d}/{EPOCHS}] '
          f'Train Loss: {t_loss:.4f} Acc: {t_acc:.2f}% | '
          f'Val Loss: {v_loss:.4f} Acc: {v_acc:.2f}% | '
          f'Time: {elapsed:.0f}s')

    # ── Save best model ──
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        patience_ctr = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': v_acc,
            'class_names': class_names
        }, '/kaggle/working/vit_best.pth')
        print(f'  ✓ Best model saved! Val Acc: {v_acc:.2f}%')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping triggered at epoch {epoch+1}')
            break

print(f'\nTraining complete! Best Val Accuracy: {best_val_acc:.2f}%')

# Save training history
with open('/kaggle/working/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

## Step 6: Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json

with open('/kaggle/working/training_history.json') as f:
    history = json.load(f)

epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ViT Fine-tuning — Training Curves', fontsize=14, fontweight='bold')

# Loss curve
axes[0].plot(epochs_ran, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs_ran, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs_ran, history['train_acc'], 'b-o', label='Train Acc', markersize=4)
axes[1].plot(epochs_ran, history['val_acc'],   'r-o', label='Val Acc',   markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved!')

## Step 7: Final Evaluation on Test Set

In [ ]:
import torch
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import time
from collections import OrderedDict

# 1. Load the raw checkpoint
checkpoint = torch.load('/kaggle/working/vit_best.pth')
state_dict = checkpoint['model_state_dict']

# 2. Strip the 'module.' prefix if it exists (Fixes Multi-GPU loading issues)
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    new_state_dict[name] = v

# 3. Load the cleaned weights and set to eval mode
model.load_state_dict(new_state_dict)
model.eval()

print(f'Loaded best model from epoch {checkpoint["epoch"]+1}')
print(f'Best validation accuracy: {checkpoint["val_acc"]:.2f}%')

# Test evaluation
all_preds, all_labels = [], []
test_correct, test_total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images).logits
        preds   = outputs.argmax(dim=1)
        test_correct += (preds == labels).sum().item()
        test_total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = test_correct / test_total * 100
print(f'\n===== TEST SET RESULTS =====')
print(f'Test Accuracy: {test_acc:.2f}%')
print(f'\nPer-class Report:')
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

In [ ]:
# FPS Benchmark
import time
import numpy as np

model.eval()
dummy_input = torch.zeros(1, 3, 224, 224).to(device)

# Warmup
with torch.no_grad():
    for _ in range(20):
        _ = model(dummy_input).logits

# Benchmark
times = []
with torch.no_grad():
    for _ in range(200):
        t0 = time.perf_counter()
        _ = model(dummy_input).logits
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

avg_latency = np.mean(times) * 1000
fps = 1.0 / np.mean(times)

print(f'\n===== INFERENCE BENCHMARK =====')
print(f'Average Latency : {avg_latency:.2f} ms')
print(f'FPS             : {fps:.1f}')
print(f'\n--- Paper-ready result ---')
print(f'ViT-Base | Test Acc: {test_acc:.2f}% | FPS: {fps:.1f} | Latency: {avg_latency:.2f}ms')

In [ ]:
# Confusion Matrix (top 15 classes for readability)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Use top 15 most common classes
from collections import Counter
top_classes_idx = [i for i, _ in Counter(all_labels).most_common(15)]
top_class_names = [class_names[i][:20] for i in top_classes_idx]

filtered_preds  = [p for p, l in zip(all_preds, all_labels) if l in top_classes_idx]
filtered_labels = [l for l in all_labels if l in top_classes_idx]

# Remap to 0..14
remap = {old: new for new, old in enumerate(top_classes_idx)}
filtered_preds  = [remap.get(p, -1) for p in filtered_preds]
filtered_labels = [remap[l] for l in filtered_labels]
filtered_preds  = [p if p != -1 else 0 for p in filtered_preds]

cm = confusion_matrix(filtered_labels, filtered_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=top_class_names,
            yticklabels=top_class_names,
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('ViT-Base — Confusion Matrix (Top 15 Classes)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved!')

## Step 8: Visualize Attention Maps

**This is your most impressive paper figure!**

Attention maps show WHAT the ViT is actually looking at when it makes a prediction. Unlike CNNs (black box), ViT lets you visualize exactly which patches it attended to — you can literally see it focusing on the diseased regions of the leaf.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random
from collections import OrderedDict
from transformers import ViTForImageClassification

# Re-initialize model with output_attentions=True
attn_model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    output_attentions=True 
).to(device)

# --- FIX: Clean DataParallel Checkpoint Prefix ---
checkpoint = torch.load('/kaggle/working/vit_best.pth')
state_dict = checkpoint['model_state_dict']
new_state_dict = OrderedDict()

for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    new_state_dict[name] = v

attn_model.load_state_dict(new_state_dict)
attn_model.eval()
print('Attention model loaded successfully (Multi-GPU prefix removed)!')

def get_attention_map(model, image_tensor, device):
    with torch.no_grad():
        outputs = model(
            image_tensor.unsqueeze(0).to(device),
            output_attentions=True
        )

    last_layer_attn = outputs.attentions[-1][0]
    attn = last_layer_attn.mean(dim=0)
    cls_attn = attn[0, 1:].cpu().numpy()

    grid_size = int(cls_attn.shape[0] ** 0.5)
    attn_map = cls_attn.reshape(grid_size, grid_size)
    return attn_map

random.seed(123)
sample_indices = random.sample(range(len(test_dataset)), min(6, len(test_dataset)))

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle('ViT Attention Maps — What the Model Focuses On', fontsize=14, fontweight='bold')

for col, idx in enumerate(sample_indices):
    image_tensor, label_idx = test_dataset[idx]
    label = class_names[label_idx]

    orig_img_path = test_samples[idx][0]
    orig_img = Image.open(orig_img_path).convert('RGB').resize((224, 224))

    attn_map = get_attention_map(attn_model, image_tensor, device)

    with torch.no_grad():
        logits = attn_model(image_tensor.unsqueeze(0).to(device)).logits
        pred_idx = logits.argmax(dim=1).item()
        pred_label = class_names[pred_idx]
        conf = torch.softmax(logits, dim=1).max().item()

    correct = '✓' if pred_idx == label_idx else '✗'

    axes[0][col].imshow(orig_img)
    axes[0][col].set_title(f'{label[:18]}', fontsize=7, fontweight='bold')
    axes[0][col].axis('off')

    attn_resized = np.array(Image.fromarray(attn_map).resize((224, 224), Image.BICUBIC))
    attn_norm = (attn_resized - attn_resized.min()) / (attn_resized.max() - attn_resized.min() + 1e-8)
    axes[1][col].imshow(orig_img, alpha=0.5)
    axes[1][col].imshow(attn_norm, cmap='jet', alpha=0.6)
    axes[1][col].set_title(f'{correct} {pred_label[:15]}\n{conf:.0%}', fontsize=7)
    axes[1][col].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Attention maps saved!')

## Step 9: Save All Assets

In [ ]:
import json
import shutil
from pathlib import Path

save_dir = Path('/kaggle/working/vit_final_assets')
save_dir.mkdir(exist_ok=True)

# Save metrics
final_metrics = {
    'model':          'ViT-Base-Patch16-224',
    'dataset':        'PlantDoc + PlantVillage (combined)',
    'task':           'Crop Leaf Disease Classification',
    'num_classes':    NUM_CLASSES,
    'test_accuracy':  round(test_acc, 2),
    'fps':            round(fps, 1),
    'latency_ms':     round(avg_latency, 2),
    'best_val_acc':   round(best_val_acc, 2),
    'classes':        class_names,
    'architecture': {
        'patch_size':   16,
        'hidden_dim':   768,
        'layers':       12,
        'heads':        12,
        'parameters':   '86M'
    }
}

with open(save_dir / 'metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

# Copy all assets
for fname in ['vit_best.pth', 'training_curves.png',
              'confusion_matrix.png', 'attention_maps.png',
              'sample_images.png', 'training_history.json',
              'class_names.json']:
    src = Path('/kaggle/working') / fname
    if src.exists():
        shutil.copy(src, save_dir / fname)

# Zip everything
shutil.make_archive('/kaggle/working/vit_assets', 'zip', save_dir)

print('All assets saved!')
print(json.dumps(final_metrics, indent=2))
print('\nDownload vit_assets.zip from the Kaggle output panel')

## Changes from v1 (PlantDoc only)

| Change | Why |
|--------|-----|
| Added PlantVillage dataset | Boosts image count for hard classes (blight, bacterial spot, septoria) from ~50 to ~1000+ each |
| Class name mapping | PlantVillage uses `Tomato___Early_blight` format — mapped to PlantDoc names |
| Combined loader | Loads both datasets and merges into single `all_samples` list |
| Same stratified split | 80/10/10 split applied across the combined pool |

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `PlantDoc not found` | Add `abdulhasibuddin/plant-doc-dataset` via + Add Data |
| `PlantVillage not found` | Add `abdallahalidev/plantvillage-dataset` via + Add Data |
| `CUDA out of memory` | Reduce batch size to 16 in DataLoader |
| PlantVillage path wrong | Print `list(PLANTVILLAGE_BASE.iterdir())` to find the right subfolder |
| Session timeout | Use **Save & Run All** for background execution |
| Low mapped count | Check that `color/` subfolder exists inside PlantVillage |

---
**Team 179 | ViT v2 — PlantDoc + PlantVillage | B.Tech Research Project**

Expected improvement over v1:
- Test Accuracy: **78–86%** (vs 72.27% in v1)
- Blight classes: **60–75%** (vs 45–56% in v1)
- Bacterial/Septoria: **55–70%** (vs 33–36% in v1)